In [10]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
model = AutoModelForCausalLM.from_pretrained("model/Qwen3-0.6B")
tokenizer = AutoTokenizer.from_pretrained("model/Qwen3-0.6B")
# 1、加载数据集
train_data = load_dataset("json",data_files={"train":"data/keywords_data_train.jsonl","test":"data/keywords_data_test.jsonl"})

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 1341.40it/s, Materializing param=model.norm.weight]                              
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [11]:
train_data

DatasetDict({
    train: Dataset({
        features: ['conversation_id', 'category', 'conversation', 'dataset'],
        num_rows: 49500
    })
    test: Dataset({
        features: ['conversation_id', 'category', 'conversation', 'dataset'],
        num_rows: 500
    })
})

In [12]:
# 2、处理数据集的格式
from typing  import List,Dict
def convert_data_format(examples:Dict[str, List]):
    """
    将数据集处理出 {"messages":[xx]}
    """
    coversation:List[List] = examples["conversation"]
    all_examples_messages:List = []

    for example in coversation:
        message_list = []
        example_message:Dict[str,str]=example[0]
        message_list.append({"role":"user","content":example_message["human"]})
        message_list.append({"role":"assistant","content":example_message["assistant"]})
        all_examples_messages.append(message_list)

    return {"messages":all_examples_messages}

mapped_train_data = train_data.map(convert_data_format,batched=True,remove_columns=['conversation_id', 'category', 'conversation', 'dataset'])

In [17]:
# 扩展：查看数据集的长度分布情况
def get_data_length(examples:Dict[str,list]):
    """
    获取到数据，经过apply chat template之后的长度
    """
    messages_lists = examples["messages"]
    length_list = []
    for message_list in messages_lists:
        result = tokenizer.apply_chat_template(message_list,tokenize=True)
        length_list.append(len(result["input_ids"]))
    return {"length":length_list}

data_with_length = mapped_train_data.map(get_data_length,batched=True,remove_columns=["messages"])

Map: 100%|██████████| 500/500 [00:00<00:00, 2949.78 examples/s]


In [19]:
res = data_with_length["train"].to_pandas()
res

,length
0,139
1,530
2,118
3,173
4,108
...,...
49495,238
49496,184
49497,325
49498,143


In [22]:
res["length"].quantile(0.999)

np.float64(654.5029999999897)

In [13]:
mapped_train_data["train"][0]

{'messages': [{'content': '高氟铍矿石在熔炼过程中配入氢氧化铝来脱除其中的氟.结果表明,在配入5％Na2CO3、9.3％Al(OH)3、1400～1500℃熔炼20 min的情况下,BeO回收率达到96％以上,脱氟效果良好(铍玻璃F/BeO能控制在15％以内).为高氟铍矿石的工业应用探索出新的冶炼途径.\n找出上文中的关键词',
   'role': 'user'},
  {'content': '高氟铍矿;配料;熔炼;回收率;脱氟率', 'role': 'assistant'}]}

In [25]:
# 3、构造SFTConfig实例
from trl.trainer.sft_config import SFTConfig
import os
os.environ["TENSORBOARD_LOGGING_DIR"] = "logs/04_trl_demo"
sft_config = SFTConfig(
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    max_steps=1000,
    num_train_epochs=1,
    logging_strategy="steps",
    logging_steps=100,
    report_to="tensorboard",
    learning_rate=3e-5,
    lr_scheduler_type="cosine",
    # warmup_ratio=0.1,
    warmup_steps=0.1,
    eval_strategy="steps",
    eval_steps=200,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    load_best_model_at_end=True,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,
    output_dir="./finetuned/04_trl_demo",
    bf16=True,
    gradient_checkpointing=True,
    activation_offloading=False,
    max_length=700,
    assistant_only_loss=True,
    chat_template_path="./chat_template.jinja"
)

# 4、构造一个SFTTrainer的实例
from trl.trainer.sft_trainer import SFTTrainer
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=mapped_train_data["train"],
    eval_dataset=mapped_train_data["test"],
    processing_class=tokenizer
)


Truncating eval dataset: 100%|██████████| 500/500 [00:00<00:00, 29345.57 examples/s]


In [30]:
train_dataloader = trainer.get_train_dataloader()
for batch_data in train_dataloader:
    result = tokenizer.decode(batch_data["input_ids"])
    print(result[0])
    break

<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 03 July 2026
Reasoning Mode: /think

## Custom Instructions

You are a helpful AI assistant named SmolLM, trained by Hugging Face. Your role as an assistant involves thoroughly exploring questions through a systematic thinking process before providing the final precise and accurate solutions. This requires engaging in a comprehensive cycle of analysis, summarizing, exploration, reassessment, reflection, backtracking, and iteration to develop well-considered thinking process. Please structure your response into two main sections: Thought and Solution using the specified format: <think> Thought section </think> Solution section. In the Thought section, detail your reasoning process in steps. Each step should include detailed considerations such as analysing questions, summarizing relevant findings, brainstorming new ideas, verifying the accuracy of the current steps, refining any errors, and revisiting previous 